# Azure Blob Storage con Python

_Subir, descargar y listar blobs usando el SDK oficial `azure-storage-blob`_

**Módulo 4 — MLOps y Cloud** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

## 1. Conceptos: cuenta → contenedor → blob

La jerarquía de Azure Storage es exactamente la misma que en S3 (cuenta = bucket-owner, contenedor = bucket, blob = objeto):

```
Storage Account ─── dsrpstorage         (único globalmente, minúsculas, sin guiones)
    └── Container ─── dsrp-modulo4      (parecido a un bucket, namespace plano)
          ├── Blob ─── raw/telco.csv    (los "/" son solo del nombre, no carpetas reales)
          └── Blob ─── models/telco.pkl
```

**Tipos de blob:**
- **Block blob** (99% de los casos): archivos arbitrarios.
- **Append blob**: logs (solo se puede añadir al final).
- **Page blob**: VHDs / discos virtuales.

En este módulo solo usamos **block blobs**.

## 2. Crear la cuenta y el contenedor (una sola vez)

Antes de poder leer/escribir, necesitas la cuenta de storage. Lo más rápido es por **Azure CLI**:

```bash
# 1. Login
az login

# 2. Variables
RG=dsrp-modulo4-rg
LOCATION=eastus
ACCOUNT=dsrpstorage$RANDOM   # tiene que ser único globalmente
CONTAINER=dsrp-modulo4

# 3. Resource group
az group create -n $RG -l $LOCATION

# 4. Storage account (Standard LRS = barato y suficiente para clase)
az storage account create \
    -n $ACCOUNT -g $RG -l $LOCATION \
    --sku Standard_LRS --kind StorageV2

# 5. Connection string (guárdala en .env)
az storage account show-connection-string -n $ACCOUNT -g $RG -o tsv

# 6. Contenedor
az storage container create --name $CONTAINER --account-name $ACCOUNT
```

Guarda la salida del paso 5 en tu `.env` (o exporta la variable directamente):

```bash
AZURE_STORAGE_CONNECTION_STRING="DefaultEndpointsProtocol=https;AccountName=...;AccountKey=...;EndpointSuffix=core.windows.net"
AZURE_STORAGE_CONTAINER="dsrp-modulo4"
```

> Si Terraform del paso 5 del módulo provisiona también la cuenta, puedes saltarte estos comandos.

## 3. Instalar el SDK

```bash
uv add azure-storage-blob azure-identity
```

- `azure-storage-blob` → cliente de blobs.
- `azure-identity` → opcional, permite autenticarse con `az login` en lugar de connection string.

In [1]:
# Imports y configuración
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient, ContentSettings

load_dotenv()  # carga .env de la raíz del repo

CONN_STR = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER = os.environ.get("AZURE_STORAGE_CONTAINER", "dsrp-modulo4")

DATA_DIR = Path("../data")
TELCO_CSV = DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

print(f"Container: {CONTAINER}")
print(f"CSV local: {TELCO_CSV}  (exists={TELCO_CSV.exists()})")

Container: dsrp-modulo4-2026
CSV local: ../data/WA_Fn-UseC_-Telco-Customer-Churn.csv  (exists=True)


## 4. Cliente y verificación del contenedor

El `BlobServiceClient` es el punto de entrada: a partir de él se obtienen clientes de contenedor y de blob individual.

In [2]:
service = BlobServiceClient.from_connection_string(CONN_STR)
container_client = service.get_container_client(CONTAINER)

# Crearlo si no existe (idempotente)
if not container_client.exists():
    container_client.create_container()
    print(f"Container '{CONTAINER}' creado.")
else:
    print(f"Container '{CONTAINER}' ya existe.")

# Propiedades de la cuenta
props = service.get_account_information()
print(f"Account kind: {props['account_kind']} | SKU: {props['sku_name']}")

Container 'dsrp-modulo4-2026' ya existe.
Account kind: StorageV2 | SKU: Standard_LRS


## 5. WRITE — subir el CSV de Telco como blob

El patrón es: abre el archivo en modo binario, llama a `upload_blob`. El `overwrite=True` es importante: por defecto Azure falla si el blob ya existe.

Marcamos el `content_type` para que el portal lo muestre como CSV.

In [3]:
BLOB_RAW = "raw/telco_churn.csv"

blob_client = container_client.get_blob_client(BLOB_RAW)

with TELCO_CSV.open("rb") as f:
    blob_client.upload_blob(
        f,
        overwrite=True,
        content_settings=ContentSettings(content_type="text/csv"),
    )

props = blob_client.get_blob_properties()
print(f"Subido: {BLOB_RAW}")
print(f"  size  : {props.size:,} bytes")
print(f"  etag  : {props.etag}")
print(f"  url   : {blob_client.url}")

Subido: raw/telco_churn.csv
  size  : 977,501 bytes
  etag  : "0x8DEBC5C98C490BE"
  url   : https://dsrp2026datab9ef9929.blob.core.windows.net/dsrp-modulo4-2026/raw/telco_churn.csv


### 5.1 Variantes útiles de upload

- **Texto/string directamente**: `blob_client.upload_blob("hola mundo", overwrite=True)`.
- **DataFrame en memoria** (sin tocar disco):

  ```python
  import io
  buf = io.BytesIO()
  df.to_csv(buf, index=False)
  buf.seek(0)
  blob_client.upload_blob(buf, overwrite=True)
  ```
- **Archivos grandes**: el SDK ya hace upload en bloques de 4 MB en paralelo; no necesitas streaming manual.

## 6. LIST — ver qué hay en el contenedor

In [4]:
print(f"Blobs en '{CONTAINER}':\n")
for blob in container_client.list_blobs():
    print(f"  - {blob.name:<40} {blob.size:>10,} bytes   {blob.last_modified:%Y-%m-%d %H:%M}")

Blobs en 'dsrp-modulo4-2026':

  - raw/telco_churn.csv                         977,501 bytes   2026-05-28 01:58


### 6.1 Filtrar por prefijo ("carpetas" virtuales)

Recuerda: los `/` en los nombres de blobs son solo convención. No hay carpetas reales — pero el SDK te deja filtrar por prefijo, que es lo que necesitas.

In [5]:
raw_blobs = list(container_client.list_blobs(name_starts_with="raw/"))
print(f"Blobs bajo 'raw/': {len(raw_blobs)}")
for b in raw_blobs:
    print(f"  - {b.name}")

Blobs bajo 'raw/': 1
  - raw/telco_churn.csv


## 7. READ — descargar el blob a un DataFrame

Hay dos formas:

1. **Descargar a un archivo temporal** y leerlo con pandas (simple, recomendable para >100 MB).
2. **Leer directamente en memoria** vía `download_blob().readall()` (más rápido, OK para < ~1 GB).

In [6]:
# Opción 2: leer en memoria → pandas
import io
import pandas as pd

data = container_client.get_blob_client(BLOB_RAW).download_blob().readall()
df = pd.read_csv(io.BytesIO(data))

print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)[:6]}...")
df.head(3)

Shape: (7043, 21)
Columnas: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure']...


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [7]:
# Opción 1: descargar a disco (útil cuando el dataset no cabe en RAM)
out_path = Path("/tmp/telco_descargado.csv")
with out_path.open("wb") as f:
    f.write(container_client.get_blob_client(BLOB_RAW).download_blob().readall())

print(f"Descargado en {out_path}  ({out_path.stat().st_size:,} bytes)")

Descargado en /tmp/telco_descargado.csv  (977,501 bytes)


## 8. Helpers reutilizables

En el resto del módulo (Docker, Terraform) vamos a llamar a Blob Storage varias veces. Vale la pena tener tres funciones tipo "librería":

In [ ]:
def upload_file(local_path: Path | str, blob_name: str, container: str = CONTAINER) -> str:
    """Sube un archivo local. Devuelve la URL del blob."""
    bc = service.get_blob_client(container=container, blob=blob_name)
    with open(local_path, "rb") as f:
        bc.upload_blob(f, overwrite=True)
    return bc.url


def download_file(blob_name: str, local_path: Path | str, container: str = CONTAINER) -> Path:
    """Descarga un blob al disco local. Devuelve el path."""
    bc = service.get_blob_client(container=container, blob=blob_name)
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    with local_path.open("wb") as f:
        f.write(bc.download_blob().readall())
    return local_path


def list_blobs(prefix: str | None = None, container: str = CONTAINER) -> list[str]:
    """Lista los nombres de blobs (opcionalmente bajo un prefijo)."""
    cc = service.get_container_client(container)
    it = cc.list_blobs(name_starts_with=prefix) if prefix else cc.list_blobs()
    return [b.name for b in it]


print("Helpers listos:", upload_file.__name__, download_file.__name__, list_blobs.__name__)

In [ ]:
# Demo end-to-end de los helpers
list_blobs(prefix="raw/")

## 9. Borrar (cleanup)

Útil al final del módulo para no acumular costos (aunque para un CSV de 1 MB la cuenta mensual es < $0.01).

```python
# Borrar un blob
container_client.delete_blob("raw/telco_churn.csv")

# Borrar todo el contenedor
container_client.delete_container()
```

Solo descoméntalo cuando hayas terminado. En el flujo del módulo, el blob `raw/telco_churn.csv` lo va a leer el contenedor Docker del paso 3, así que **no lo borres todavía**.

## 10. Seguridad — qué NO hacer

- **No commitees la connection string.** El `.env` ya está en `.gitignore`, mantenlo así.
- **No uses la account key en producción.** Para apps reales, usa `azure-identity` con Managed Identity o un SAS token con permisos mínimos y expiración corta.
- **No hagas público el contenedor** salvo que el dataset realmente sea público. Por defecto los contenedores son privados (`--public-access off`).

Para el módulo, la connection string en `.env` es suficiente porque es un ambiente de aprendizaje.

---

**Próximo paso**: el contenedor del CSV está listo en Azure. Ve a `docker-training/` para construir la imagen que entrena el modelo y sube el `.pkl` resultante al mismo contenedor, bajo `models/`.